# COS40007 — Structural Defect Detection (Kaggle)

Train and evaluate all four detectors on the cleaned 5-class dataset.

**Setup:**
1. Upload `defect_dataset.zip` as a Kaggle dataset (e.g. `defect-dataset-v2`).
   Add it via **+ Add data** in the notebook editor.
2. Accelerator → **GPU T4 x2** (or P100)
3. **Run all** — results zip to `/kaggle/working/runs_results.zip`
4. **Save Version** (top-right) before the session ends to commit the output.

Classes: `cracks, spalling, corrosion, potholes, paint_degradation`

## Part 0 — Environment Setup

In [ ]:
# Dependencies — install only if missing
import importlib.util, subprocess, sys
for pkg, mod in [('ultralytics', 'ultralytics'), ('torchmetrics', 'torchmetrics')]:
    if importlib.util.find_spec(mod) is None:
        print('installing', pkg, '...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
print('Dependencies ready.')

In [ ]:
# Locate the dataset zip added via '+ Add data' and unzip to /kaggle/working.
import os, sys, zipfile, shutil, glob
from pathlib import Path

IN_COLAB  = False
IN_KAGGLE = os.path.exists('/kaggle/working')
WORK_DIR  = Path('/kaggle/working') if IN_KAGGLE else Path('..').resolve()

if IN_KAGGLE:
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    if not (WORK_DIR / 'data').exists():
        # Kaggle datasets are mounted under /kaggle/input/<dataset-slug>/
        candidates = glob.glob('/kaggle/input/**/defect_dataset.zip', recursive=True)
        assert candidates, (
            'defect_dataset.zip not found under /kaggle/input/. '
            'Add the dataset via + Add data and re-run.')
        zip_path = candidates[0]
        print('Found zip:', zip_path)
        print('Copying to working dir...')
        local_zip = WORK_DIR / 'defect_dataset.zip'
        shutil.copy(zip_path, local_zip)
        print('Unzipping (forward-slash safe)...')
        with zipfile.ZipFile(local_zip) as z:
            for member in z.infolist():
                fixed = member.filename.replace('\\', '/')
                target = WORK_DIR / fixed
                if fixed.endswith('/'):
                    target.mkdir(parents=True, exist_ok=True)
                else:
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with z.open(member) as src, open(target, 'wb') as dst:
                        shutil.copyfileobj(src, dst)
        local_zip.unlink()  # free space
        print('Done.')
    else:
        print('data/ already present — skipping unzip.')
else:
    print('Local run — using existing project data/.')

In [ ]:
# Paths
PROJECT_ROOT = WORK_DIR
DATA_DIR     = PROJECT_ROOT / 'data'
RUNS_DIR     = PROJECT_ROOT / 'runs'
RUNS_DIR.mkdir(parents=True, exist_ok=True)
assert DATA_DIR.exists(), 'data/ not found at %s' % DATA_DIR
print('DATA_DIR:', DATA_DIR)
print('RUNS_DIR:', RUNS_DIR)

In [ ]:
# Imports, classes, device, and data.yaml (rewritten with the correct absolute path)
import torch, time, csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

CLASSES      = ['cracks', 'spalling', 'corrosion', 'potholes', 'paint_degradation']
CLASS_COLORS = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
NUM_CLASSES  = len(CLASSES) + 1          # +1 background (torchvision / SSD convention)
IMAGE_EXTS   = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

def write_data_yaml():
    (DATA_DIR / 'data.yaml').write_text(
        'path: %s\ntrain: images/train\nval: images/val\ntest: images/test\n'
        'nc: %d\nnames: %s\n' % (DATA_DIR.as_posix(), len(CLASSES), CLASSES))

write_data_yaml()
DATA_YAML = str(DATA_DIR / 'data.yaml')
print('data.yaml ->', DATA_YAML)

# severity from bbox area fraction (Low <5%, Medium 5-20%, High >20%)
def severity_label(bw, bh):
    a = bw * bh * 100
    return 'Low' if a < 5 else ('Medium' if a <= 20 else 'High')


In [ ]:
# Hyperparameters — auto-scaled to the GPU. FAIRNESS: all three models train for the
# SAME number of epochs; only architecture-appropriate optimisers / anchors differ.
GPU_MEM_GB = (torch.cuda.get_device_properties(0).total_memory / 1e9) if torch.cuda.is_available() else 0.0
BIG_GPU    = GPU_MEM_GB >= 12
ON_WINDOWS = os.name == 'nt'

QUICK_TEST = False                         # True -> 3 epochs each, for a fast pipeline smoke-test
EPOCHS     = 3 if QUICK_TEST else 110      # identical epoch budget for YOLOv11s, YOLOv8s, SSDLite

# YOLO (Ultralytics, anchor-free)
YOLO_BATCH = 16 if BIG_GPU else 8
LR         = 1e-3                          # AdamW + cosine tolerates a higher initial LR
IMG_SIZE   = 640
WORKERS    = 0 if ON_WINDOWS else 2        # Windows dataloader workers must be 0
USE_AMP    = torch.cuda.is_available()

# SSDLite (torchvision, anchor-based)
SSD_BATCH  = 8 if BIG_GPU else 2
SSD_IMG    = 640 if BIG_GPU else 320       # 640 "if possible"; 320 is native and fits a 4 GB GPU
SSD_LR     = 5e-4
ANCHOR_AR  = (2, 3, 5)                     # boxes {1/5..5} vs default {1/3..3} -> fits long/narrow defects

print('GPU mem: %.1f GB | BIG_GPU=%s | Windows=%s' % (GPU_MEM_GB, BIG_GPU, ON_WINDOWS))
print('EPOCHS (all 3 models): %d%s' % (EPOCHS, '   [QUICK_TEST]' if QUICK_TEST else ''))
print('YOLO: batch=%d img=%d lr=%.4f  AdamW + cosine, AMP=%s' % (YOLO_BATCH, IMG_SIZE, LR, USE_AMP))
print('SSD : batch=%d img=%d lr=%.4f  SGD mom0.9, anchors_ar=%s' % (SSD_BATCH, SSD_IMG, SSD_LR, ANCHOR_AR))

## Part 1 — Dataset Integrity Check (before training)

Verifies the cleaned dataset before any training so the results are trustworthy:
1. every image has a label and every label an image,
2. every label has at least one valid box,
3. no box has width = 0 or height = 0,
4. all class IDs are in range 0–4,
5. the train / val / test splits are disjoint (no cross-split leakage), and
6. bounding-box sizes are sane (corrosion in particular).

The dataset was already cleaned by `scripts/clean_dataset.py`, so these cells should report
**0 issues** — they exist to *prove* it, and to re-flag problems if the data is ever
regenerated. Inspection only: this notebook never edits labels.

In [ ]:
# 1.1 Integrity: labels<->images, >=1 valid box, class-id range, w/h > 0
def split_paths(split):
    img_dir = DATA_DIR / 'images' / split
    lbl_dir = DATA_DIR / 'labels' / split
    imgs = sorted(p for p in img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS) if img_dir.exists() else []
    return img_dir, lbl_dir, imgs

issues = {'missing_label': [], 'empty_label': [], 'no_valid_box': [],
          'degenerate_box': [], 'bad_class_id': [], 'orphan_label': []}

for split in ('train', 'val', 'test'):
    img_dir, lbl_dir, imgs = split_paths(split)
    img_stems = {p.stem for p in imgs}
    for p in imgs:
        lbl = lbl_dir / (p.stem + '.txt')
        if not lbl.exists():
            issues['missing_label'].append('%s/%s' % (split, p.name)); continue
        lines = [ln for ln in lbl.read_text().splitlines() if ln.strip()]
        if not lines:
            issues['empty_label'].append('%s/%s' % (split, lbl.name)); continue
        valid = 0
        for ln in lines:
            parts = ln.split()
            if len(parts) != 5:
                continue
            cid = int(parts[0]); cx, cy, bw, bh = map(float, parts[1:])
            if not (0 <= cid < len(CLASSES)):
                issues['bad_class_id'].append('%s/%s -> %d' % (split, lbl.name, cid)); continue
            if bw <= 0 or bh <= 0:
                issues['degenerate_box'].append('%s/%s -> %s' % (split, lbl.name, ln)); continue
            valid += 1
        if valid == 0:
            issues['no_valid_box'].append('%s/%s' % (split, lbl.name))
    if lbl_dir.exists():
        for lbl in lbl_dir.glob('*.txt'):
            if lbl.stem not in img_stems:
                issues['orphan_label'].append('%s/%s' % (split, lbl.name))

print('Dataset integrity check'); print('-' * 50)
for k, v in issues.items():
    print('%s %-16s : %d' % ('OK ' if not v else 'XX ', k, len(v)))
    for item in v[:5]:
        print('       -', item)
    if len(v) > 5:
        print('       ... (%d more)' % (len(v) - 5))
print('-' * 50); print('TOTAL ISSUES:', sum(len(v) for v in issues.values()), '(0 = clean)')

In [ ]:
# 1.2 Split separation + cross-split duplicate (perceptual a-hash) leakage check
def ahash(path, hash_size=8):
    try:
        img = Image.open(path).convert('L').resize((hash_size, hash_size), Image.BILINEAR)
    except Exception:
        return None
    arr = np.asarray(img, dtype=np.float32)
    bits = (arr > arr.mean()).flatten()
    h = 0
    for b in bits:
        h = (h << 1) | int(b)
    return h

split_imgs = {s: split_paths(s)[2] for s in ('train', 'val', 'test')}
stems = {s: {p.stem for p in split_imgs[s]} for s in split_imgs}
print('Exact stem overlaps (should all be 0):')
print('  train&val :', len(stems['train'] & stems['val']))
print('  train&test:', len(stems['train'] & stems['test']))
print('  val&test  :', len(stems['val'] & stems['test']))

hashes = {s: {} for s in split_imgs}
for s in split_imgs:
    for p in split_imgs[s]:
        h = ahash(p)
        if h is not None:
            hashes[s].setdefault(h, []).append(p.name)

def cross(a, b):
    common = set(hashes[a]) & set(hashes[b])
    return [(hashes[a][h][0], hashes[b][h][0]) for h in common]

print('\nCross-split perceptual-hash collisions (should all be 0):')
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    pairs = cross(a, b)
    print('  %-11s: %d' % ('%s-%s' % (a, b), len(pairs)))
    for x, y in pairs[:5]:
        print('       ', x, '<->', y)

In [ ]:
# 1.3 Bounding-box size audit (focus: corrosion) — flag boxes < 0.5% of image area
AREA_THRESH = 0.005
flagged = []
per_class_tiny = {c: 0 for c in CLASSES}
for split in ('train', 'val', 'test'):
    img_dir, lbl_dir, imgs = split_paths(split)
    for p in imgs:
        lbl = lbl_dir / (p.stem + '.txt')
        if not lbl.exists():
            continue
        for ln in lbl.read_text().splitlines():
            parts = ln.split()
            if len(parts) != 5:
                continue
            cid = int(parts[0]); cx, cy, bw, bh = map(float, parts[1:])
            if not (0 <= cid < len(CLASSES)):
                continue
            area = bw * bh
            if 0 < area < AREA_THRESH:
                per_class_tiny[CLASSES[cid]] += 1
                if CLASSES[cid] == 'corrosion':
                    flagged.append((split, p, cx, cy, bw, bh, area))

print('Boxes smaller than %.1f%% of image area, by class:' % (AREA_THRESH * 100))
for c in CLASSES:
    print('  %-18s: %d' % (c, per_class_tiny[c]))
print('\nFlagged CORROSION boxes for visual review:', len(flagged))

show = flagged[:6]
if show:
    fig, axes = plt.subplots(1, len(show), figsize=(4 * len(show), 4))
    if len(show) == 1:
        axes = [axes]
    for ax, (split, p, cx, cy, bw, bh, area) in zip(axes, show):
        img = Image.open(p).convert('RGB'); W, H = img.size
        ax.imshow(img)
        ax.add_patch(patches.Rectangle(((cx - bw / 2) * W, (cy - bh / 2) * H), bw * W, bh * H,
                     lw=2, edgecolor='#2ecc71', facecolor='none'))
        ax.set_title('%s\narea=%.3f%%' % (p.name, area * 100), fontsize=8); ax.axis('off')
    plt.suptitle('Flagged corrosion boxes — review; enlarge only if clearly wrong')
    plt.tight_layout(); plt.savefig(RUNS_DIR / 'flagged_corrosion_examples.png', dpi=120); plt.show()
else:
    print('No tiny corrosion boxes — dataset already cleaned.')

In [ ]:
# 1.4 Per-split counts + class-distribution chart (documents balance for the report)
def count_bbox_per_class(split):
    counts = {c: 0 for c in CLASSES}
    for lbl in (DATA_DIR / 'labels' / split).glob('*.txt'):
        for line in lbl.read_text().splitlines():
            parts = line.split()
            if parts:
                cid = int(parts[0])
                if 0 <= cid < len(CLASSES):
                    counts[CLASSES[cid]] += 1
    return counts

rows = [{'split': s, 'images': len(split_paths(s)[2]), **count_bbox_per_class(s)}
        for s in ('train', 'val', 'test')]
stats_df = pd.DataFrame(rows).set_index('split')
print(stats_df.to_string())

split_colors = {'train': '#34495e', 'val': '#7f8c8d', 'test': '#bdc3c7'}
tallies = {s: count_bbox_per_class(s) for s in ('train', 'val', 'test')}
fig, ax = plt.subplots(figsize=(9.5, 4.5))
bottom = [0] * len(CLASSES)
for s in ('train', 'val', 'test'):
    vals = [tallies[s][c] for c in CLASSES]
    ax.bar(CLASSES, vals, bottom=bottom, label=s, color=split_colors[s], edgecolor='white', width=0.6)
    bottom = [b + v for b, v in zip(bottom, vals)]
for i, total in enumerate(bottom):
    ax.text(i, total + 15, str(total), ha='center', va='bottom', fontsize=9, fontweight='bold')
ratio = (max(bottom) / min(bottom)) if min(bottom) else 0
ax.set_ylabel('Number of bounding boxes')
ax.set_title('Class Distribution by Split (balance ratio: %.2fx)' % ratio)
ax.legend(title='split'); plt.tight_layout()
plt.savefig(RUNS_DIR / 'class_distribution.png', dpi=150); plt.show()

In [ ]:
# 1.5 One labelled sample image per class (sanity check that boxes line up with defects)
def find_img(stem):
    for ext in IMAGE_EXTS:
        p = DATA_DIR / 'images' / 'train' / (stem + ext)
        if p.exists():
            return p
    return None

def draw_bboxes(ax, img, label_path):
    w, h = img.size
    ax.imshow(img)
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            parts = line.split()
            if len(parts) != 5:
                continue
            cls, cx, cy, bw, bh = int(parts[0]), *map(float, parts[1:])
            ax.add_patch(patches.Rectangle(((cx - bw / 2) * w, (cy - bh / 2) * h), bw * w, bh * h,
                         lw=2, edgecolor=CLASS_COLORS[cls], facecolor='none'))
    ax.axis('off')

lbl_dir = DATA_DIR / 'labels' / 'train'
by_class = {c: [] for c in range(len(CLASSES))}
for lbl in lbl_dir.glob('*.txt'):
    t = lbl.read_text().strip()
    if t:
        by_class[int(t.split()[0])].append(lbl.stem)

fig, axes = plt.subplots(1, len(CLASSES), figsize=(20, 4))
for ci, name in enumerate(CLASSES):
    ax = axes[ci]
    if by_class[ci]:
        p = find_img(by_class[ci][0])
        if p:
            draw_bboxes(ax, Image.open(p).convert('RGB'), lbl_dir / (by_class[ci][0] + '.txt'))
    ax.set_title(name)
plt.suptitle('Sample Training Images with Ground-Truth Boxes')
plt.tight_layout(); plt.savefig(RUNS_DIR / 'sample_images.png', dpi=120, bbox_inches='tight'); plt.show()

## Part 1.6 Dataset repair and verification

Repairs the dataset before training: convert/remove GIF-as-JPG, re-encode corrupt JPEGs, drop duplicate/invalid label lines, remove unopenable images. **`DRY_RUN=True` by default** (reports only). Set `DRY_RUN=False` to apply, or run `python scripts/prepare_training_data.py --apply`.

In [ ]:
from collections import Counter
from PIL import Image, ImageFile

CLASSES = ['cracks', 'spalling', 'corrosion', 'potholes', 'paint_degradation']
IMAGE_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
SPLITS = ('train', 'val', 'test')


def _inspect_image(path):
    """Return one of: 'ok', 'convert' (wrong format named .jpg), 'reencode' (truncated but
    recoverable), 'remove' (cannot open). Second value is the detected format string."""
    try:
        with Image.open(path) as im:
            fmt = (im.format or '').upper()
            im.load()  # strict decode
        if path.suffix.lower() in ('.jpg', '.jpeg') and fmt not in ('JPEG', 'MPO'):
            return 'convert', fmt          # e.g. a GIF/PNG saved with a .jpg name
        return 'ok', fmt
    except Exception:
        pass
    # lenient pass: recover truncated/corrupt images PIL can still decode
    ImageFile.LOAD_TRUNCATED_IMAGES = True
    try:
        with Image.open(path) as im:
            fmt = (im.format or '').upper()
            im.convert('RGB').load()
        return 'reencode', fmt
    except Exception:
        return 'remove', '?'
    finally:
        ImageFile.LOAD_TRUNCATED_IMAGES = False


def _save_rgb(path):
    """Re-encode an image to a clean RGB file at the same path (JPEG for .jpg/.jpeg, else PNG)."""
    ImageFile.LOAD_TRUNCATED_IMAGES = True
    try:
        with Image.open(path) as im:
            rgb = im.convert('RGB')
        if path.suffix.lower() in ('.jpg', '.jpeg'):
            rgb.save(path, 'JPEG', quality=95)
        elif path.suffix.lower() == '.png':
            rgb.save(path, 'PNG')
        else:
            rgb.save(path)
    finally:
        ImageFile.LOAD_TRUNCATED_IMAGES = False


def _clean_label_lines(text):
    """Return (kept_lines, n_duplicates, n_invalid). A line is valid when it has 5 fields,
    class_id in 0..4, x/y in [0,1], and w,h in (0,1]. Exact duplicate boxes are dropped."""
    seen, kept = set(), []
    dupes = invalid = 0
    for ln in text.splitlines():
        s = ln.strip()
        if not s:
            continue
        parts = s.split()
        if len(parts) != 5:
            invalid += 1
            continue
        try:
            cid = int(parts[0])
            cx, cy, w, h = (float(v) for v in parts[1:])
        except ValueError:
            invalid += 1
            continue
        if not (0 <= cid < len(CLASSES)):
            invalid += 1
            continue
        if not (0.0 <= cx <= 1.0 and 0.0 <= cy <= 1.0):
            invalid += 1
            continue
        if not (0.0 < w <= 1.0 and 0.0 < h <= 1.0):
            invalid += 1
            continue
        key = (cid, round(cx, 6), round(cy, 6), round(w, 6), round(h, 6))
        if key in seen:
            dupes += 1
            continue
        seen.add(key)
        kept.append(s)
    return kept, dupes, invalid


def prepare_dataset(data_dir, apply=False):
    """Repair images and labels under data_dir. Counts everything; only writes when apply=True."""
    data_dir = Path(data_dir)
    n = Counter()
    for split in SPLITS:
        img_dir = data_dir / 'images' / split
        lbl_dir = data_dir / 'labels' / split
        if not img_dir.exists():
            continue
        for img in sorted(img_dir.iterdir()):
            if img.suffix.lower() not in IMAGE_EXTS:
                continue
            n['files_checked'] += 1
            lbl = lbl_dir / (img.stem + '.txt')
            status, _ = _inspect_image(img)
            if status == 'remove':
                n['images_removed'] += 1
                if apply:
                    img.unlink(missing_ok=True)
                    lbl.unlink(missing_ok=True)
                continue
            if status in ('convert', 'reencode'):
                n['images_fixed'] += 1
                if apply:
                    _save_rgb(img)
            if lbl.exists():
                kept, dupes, invalid = _clean_label_lines(lbl.read_text())
                n['duplicate_labels_removed'] += dupes
                n['invalid_labels_removed'] += invalid
                if not kept:                       # nothing valid left -> drop the image too
                    n['images_removed'] += 1
                    if apply:
                        img.unlink(missing_ok=True)
                        lbl.unlink(missing_ok=True)
                elif (dupes or invalid) and apply:
                    lbl.write_text('\n'.join(kept) + '\n')
    return n


def verify_dataset(data_dir):
    """Print a short per-class / per-split summary (used when verify_rebuild.py is unavailable)."""
    data_dir = Path(data_dir)
    per = {c: 0 for c in range(len(CLASSES))}
    imgs = {s: 0 for s in SPLITS}
    total = 0
    for s in SPLITS:
        ld = data_dir / 'labels' / s
        if not ld.exists():
            continue
        for lp in ld.glob('*.txt'):
            imgs[s] += 1
            for ln in lp.read_text().splitlines():
                p = ln.split()
                if len(p) != 5:
                    continue
                try:
                    c = int(p[0])
                except ValueError:
                    continue
                if 0 <= c < len(CLASSES):
                    per[c] += 1
                    total += 1
    print('Verification summary')
    print('  images: ' + ', '.join('%s=%d' % (s, imgs[s]) for s in SPLITS)
          + ' (total %d)' % sum(imgs.values()))
    for c, name in enumerate(CLASSES):
        print('  %-18s %d' % (name, per[c]))
    nz = [v for v in per.values() if v]
    bal = (max(nz) / min(nz)) if nz else 0.0
    print('  total boxes: %d | balance ratio: %.2fx' % (total, bal))

# Part 1.6 driver
DRY_RUN = True   # set False to actually modify data/ (labels are git-tracked; images are not)
if DRY_RUN:
    print('DRY_RUN=True: no files were changed. Set DRY_RUN=False or run --apply to apply repairs.')
else:
    print('APPLYING changes to data/ (labels are git-tracked).')
_summary = prepare_dataset(DATA_DIR, apply=not DRY_RUN)
print('\nRepair summary')
for _k in ['files_checked', 'images_fixed', 'images_removed',
           'duplicate_labels_removed', 'invalid_labels_removed']:
    print('  %-26s %d' % (_k, _summary[_k]))
print()
verify_dataset(DATA_DIR)

## Part 2 — Training (YOLOv11s + YOLOv8s two-stage; SSDLite/RT-DETR optional)

Transfer learning, each run auto-versioned into `runs/`. YOLO via Ultralytics; SSDLite via a
two-phase torchvision loop. On a T4 the full run is ~2.5 h per YOLO model + ~1 h SSDLite.

In [ ]:
# 2.0 Shared run logger, version helper, and YOLO trainer
from ultralytics import YOLO

def log_run(name, epochs, batch, imgsz, opt, map50, map50_95, precision, recall,
            train_min, params_m, weights, notes=''):
    log_path = RUNS_DIR / 'run_log.csv'
    new = not log_path.exists()
    with open(log_path, 'a', newline='') as f:
        w = csv.writer(f)
        if new:
            w.writerow(['run_id', 'model', 'epochs', 'batch', 'imgsz', 'optimizer', 'lr',
                        'map50', 'map50_95', 'precision', 'recall',
                        'train_time_min', 'params_M', 'weights', 'notes'])
        run_id = '%s_%d' % (name, int(time.time()))
        w.writerow([run_id, name, epochs, batch, imgsz, opt, LR,
                    round(map50, 4), round(map50_95, 4), round(precision, 4), round(recall, 4),
                    round(train_min, 1), round(params_m, 2), weights, notes])
    print('Logged run:', run_id)

def next_version(subdir):
    v = 1
    while (RUNS_DIR / subdir / ('v%d' % v)).exists():
        v += 1
    return v

def train_yolo(weights_name, subdir, label):
    ver = next_version(subdir)
    print('%s -> runs/%s/v%d' % (label, subdir, ver))
    model = YOLO(weights_name)   # auto-downloads COCO weights if absent
    t0 = time.time()
    model.train(
        data=DATA_YAML, epochs=EPOCHS, imgsz=IMG_SIZE, batch=YOLO_BATCH, lr0=LR,
        optimizer='AdamW', cos_lr=True, warmup_epochs=5,
        mosaic=1.0, close_mosaic=20, degrees=5.0, scale=0.4, translate=0.08,
        fliplr=0.5, hsv_h=0.015, hsv_s=0.5, hsv_v=0.35,
        box=8.0, cls=0.4, dfl=1.7,   # nudge loss toward localization (defects = localization+recall)
        erasing=0.0,        # no random erasing - blacks out patches, kills recall on tiny/thin defects
        device=0 if torch.cuda.is_available() else 'cpu',
        project=str(RUNS_DIR / subdir), name='v%d' % ver, exist_ok=False,
        save=True, amp=USE_AMP, workers=WORKERS, verbose=True)
    mins = (time.time() - t0) / 60
    val = model.val(data=DATA_YAML)
    w = str(RUNS_DIR / subdir / ('v%d' % ver) / 'weights' / 'best.pt')
    params = sum(p.numel() for p in model.model.parameters()) / 1e6
    print('%s: mAP@0.5=%.4f mAP@0.5:0.95=%.4f (%.0f min)' % (label, val.box.map50, val.box.map, mins))
    log_run(label, EPOCHS, YOLO_BATCH, IMG_SIZE, 'AdamW', float(val.box.map50), float(val.box.map),
            float(val.box.mp), float(val.box.mr), mins, params, w, notes='small model, AdamW, cosine LR, safer aug (deg5/scale0.4/hsv-modest/no-erase), loss->loc')
    return model

print('Training helpers ready.')

## Part 2.1 Training configuration

Run switches and shared training functions used by Parts 2.2 and 2.3. **Default workflow: `baseline_640` only.** The 768 fine-tune (`RUN_FINETUNE_768`) is disabled by default — it was tested on v2 and v3 and regressed vs the 640 baseline. Enable it only for ablation experiments. `CACHE_MODE="disk"` speeds loading; `FORCE_RETRAIN=True` ignores existing checkpoints.

In [ ]:
# Part 2.1 Training configuration - enable one model per session.
RUN_YOLO11S      = True
RUN_YOLO8S       = False
RUN_SSDLITE      = False
RUN_RTDETR       = False
RUN_FINETUNE_768 = False   # optional ablation only - tested on v2/v3, regressed vs baseline_640
QUICK_TEST       = False   # True -> 3-epoch smoke test of each stage
CACHE_MODE       = "disk"  # or False if storage is limited
FORCE_RETRAIN    = False   # True -> ignore existing checkpoints and retrain from scratch
DATASET_VERSION  = "v3"    # tag written into experiment_tracker.csv for traceability
print('config: RUN_YOLO11S=%s RUN_YOLO8S=%s RUN_SSDLITE=%s RUN_RTDETR=%s RUN_FINETUNE_768=%s | '
      'QUICK_TEST=%s CACHE_MODE=%s FORCE_RETRAIN=%s DATASET_VERSION=%s'
      % (RUN_YOLO11S, RUN_YOLO8S, RUN_SSDLITE, RUN_RTDETR, RUN_FINETUNE_768,
         QUICK_TEST, CACHE_MODE, FORCE_RETRAIN, DATASET_VERSION))

import time
from pathlib import Path
import torch
from ultralytics import YOLO

TRACKER_COLS = ['dataset_version', 'model', 'stage', 'run_dir', 'imgsz', 'epochs_planned',
                'optimizer', 'lr0', 'patience', 'val_mAP50', 'val_mAP50_95',
                'precision', 'recall', 'comments']


def _ckpt_epoch(path):
    # Ultralytics sets epoch=-1 in a checkpoint once training finished (optimizer stripped).
    try:
        return int(torch.load(str(path), map_location='cpu').get('epoch', -1))
    except Exception:
        return -1


def resume_or_start(run_dir, init_weights):
    """Decide how to (re)start a stage in its own folder. Returns (action, weights).
    action in {'resume','complete','fresh'}. resume is only ever for the SAME interrupted run."""
    last = run_dir / 'weights' / 'last.pt'
    best = run_dir / 'weights' / 'best.pt'
    if FORCE_RETRAIN:
        print('  FORCE_RETRAIN=True -> fresh run (ignoring checkpoints in %s)' % run_dir.name)
        return 'fresh', init_weights
    if last.exists():
        ep = _ckpt_epoch(last)
        if ep == -1 and best.exists():
            print('  %s already complete -> validate only' % run_dir.name)
            return 'complete', best
        print('  %s interrupted (epoch %d) -> resume' % (run_dir.name, ep))
        return 'resume', last
    return 'fresh', init_weights


def _persist_base():
    mydrive = Path('/content/drive/MyDrive')          # Colab (Drive mounted)
    if mydrive.exists():
        d = mydrive / 'COS40007'
        d.mkdir(parents=True, exist_ok=True)
        return d
    kaggle = Path('/kaggle/working')                  # Kaggle output
    if kaggle.exists():
        return kaggle
    return None                                       # local - already persistent


def persist_results(subdir, run_name):
    """Back up only the just-finished run folder + the tracker/notes (not all of runs/)."""
    import shutil
    base = _persist_base()
    run_dir = RUNS_DIR / subdir / run_name
    if base is None:
        print('  local: %s already persistent' % run_dir.as_posix())
        return
    shutil.make_archive(str(base / ('%s_%s' % (subdir, run_name))), 'zip',
                        root_dir=str(run_dir.parent), base_dir=run_name)
    for f in ('experiment_tracker.csv', 'training_notes.md'):
        src = RUNS_DIR / f
        if src.exists():
            shutil.copy2(src, base / f)
    print('  backed up %s + tracker/notes -> %s' % (run_name, base))


def append_training_notes(line):
    with open(RUNS_DIR / 'training_notes.md', 'a', encoding='utf-8') as f:
        f.write(line.rstrip() + '\n')


def save_val_metrics(model, stage, run_dir, imgsz, epochs_planned, lr0, patience, val, comments=''):
    """Upsert one validation row into runs/experiment_tracker.csv (validation only, never test)."""
    import pandas as pd
    p, r = float(val.box.mp), float(val.box.mr)
    row = {'dataset_version': DATASET_VERSION, 'model': model, 'stage': stage,
           'run_dir': run_dir.as_posix(), 'imgsz': imgsz, 'epochs_planned': epochs_planned,
           'optimizer': 'AdamW', 'lr0': lr0, 'patience': patience,
           'val_mAP50': round(float(val.box.map50), 4), 'val_mAP50_95': round(float(val.box.map), 4),
           'precision': round(p, 4), 'recall': round(r, 4), 'comments': comments}
    path = RUNS_DIR / 'experiment_tracker.csv'
    df = pd.read_csv(path) if path.exists() else pd.DataFrame(columns=TRACKER_COLS)
    # add missing columns for backward compat with older CSVs
    for col in TRACKER_COLS:
        if col not in df.columns:
            df[col] = ''
    df = df[~((df['model'] == model) & (df['stage'] == stage) &
              (df.get('dataset_version', '') == DATASET_VERSION))]
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)[TRACKER_COLS]
    df.to_csv(path, index=False)
    print('  tracker: %s %s [%s] val mAP@0.5=%.4f mAP@0.5:0.95=%.4f'
          % (model, stage, DATASET_VERSION, row['val_mAP50'], row['val_mAP50_95']))
    return row


def _validate_and_record(pretty, stage, subdir, run_name, imgsz, epochs, lr0, patience):
    run_dir = RUNS_DIR / subdir / run_name
    best = run_dir / 'weights' / 'best.pt'
    val = YOLO(str(best)).val(data=DATA_YAML, split='val', imgsz=imgsz)
    save_val_metrics(pretty, stage, run_dir, imgsz, epochs, lr0, patience, val)
    append_training_notes('- %s %s: val mAP@0.5=%.4f mAP@0.5:0.95=%.4f (imgsz=%d, weights=%s)'
                          % (pretty, stage, float(val.box.map50), float(val.box.map),
                             imgsz, best.as_posix()))
    persist_results(subdir, run_name)
    return val


def train_baseline_640(init_weights, subdir, pretty):
    """Stage 1: 640 baseline (epochs=110, patience=25). Returns best.pt path, or None if interrupted."""
    run_dir = RUNS_DIR / subdir / 'baseline_640'
    epochs = 3 if QUICK_TEST else 110
    close_mosaic = 0 if QUICK_TEST else 20
    action, ckpt = resume_or_start(run_dir, init_weights)
    if action == 'complete':
        _validate_and_record(pretty, 'baseline_640', subdir, 'baseline_640', 640, epochs, 0.001, 25)
        return run_dir / 'weights' / 'best.pt'

    def _do(batch):
        YOLO(str(ckpt)).train(
            data=DATA_YAML, imgsz=640, epochs=epochs, patience=25, batch=batch,
            optimizer='AdamW', lr0=0.001, cos_lr=True, warmup_epochs=5,
            mosaic=1.0, close_mosaic=close_mosaic, degrees=5.0, scale=0.4, translate=0.08,
            fliplr=0.5, hsv_h=0.015, hsv_s=0.5, hsv_v=0.35,
            erasing=0.0, multi_scale=False, box=8.0, cls=0.4, dfl=1.7,
            save_period=5, cache=CACHE_MODE, plots=True,
            device=0 if torch.cuda.is_available() else 'cpu',
            project=str(RUNS_DIR / subdir), name='baseline_640', exist_ok=True,
            workers=WORKERS, amp=USE_AMP, verbose=True)

    batch = 16 if BIG_GPU else 8
    t0 = time.time()
    try:
        if action == 'resume':
            YOLO(str(ckpt)).train(resume=True)
        else:
            try:
                _do(batch)
            except RuntimeError as e:
                if 'out of memory' in str(e).lower() and batch > 8:
                    print('  OOM at batch %d -> retry at 8' % batch)
                    torch.cuda.empty_cache()
                    _do(8)
                else:
                    raise
    except KeyboardInterrupt:
        print('  %s baseline_640 interrupted - re-run the cell to resume from last.pt' % pretty)
        return None
    print('  %s baseline_640 finished in %.0f min' % (pretty, (time.time() - t0) / 60))
    _validate_and_record(pretty, 'baseline_640', subdir, 'baseline_640', 640, epochs, 0.001, 25)
    return run_dir / 'weights' / 'best.pt'


def finetune_768(subdir, pretty):
    """Stage 2: 768 fine-tune (epochs=40, patience=12) from the 640 best.pt. resume only within 768."""
    base_best = RUNS_DIR / subdir / 'baseline_640' / 'weights' / 'best.pt'
    if not base_best.exists():
        print('  %s: no 640 best.pt -> run the baseline first; skipping 768' % pretty)
        return None
    run_dir = RUNS_DIR / subdir / 'finetune_768'
    epochs = 3 if QUICK_TEST else 40
    action, ckpt = resume_or_start(run_dir, base_best)
    if action == 'complete':
        _validate_and_record(pretty, 'finetune_768', subdir, 'finetune_768', 768, epochs, 0.0002, 12)
        return run_dir / 'weights' / 'best.pt'

    def _do(batch):
        # plain load from the 640 best.pt, resume=False (this is a fresh fine-tune, not a resume)
        YOLO(str(base_best)).train(
            data=DATA_YAML, imgsz=768, epochs=epochs, patience=12, batch=batch,
            optimizer='AdamW', lr0=0.0002, cos_lr=True, warmup_epochs=3,
            mosaic=0.0, close_mosaic=0, degrees=3.0, scale=0.25, translate=0.05,
            fliplr=0.5, hsv_h=0.015, hsv_s=0.4, hsv_v=0.3,
            erasing=0.0, multi_scale=False, box=8.0, cls=0.4, dfl=1.7,
            save_period=5, cache=CACHE_MODE, plots=True,
            device=0 if torch.cuda.is_available() else 'cpu',
            project=str(RUNS_DIR / subdir), name='finetune_768', exist_ok=True,
            workers=WORKERS, amp=USE_AMP, verbose=True)

    batch = 8 if BIG_GPU else 4
    t0 = time.time()
    try:
        if action == 'resume':
            YOLO(str(ckpt)).train(resume=True)
        else:
            try:
                _do(batch)
            except RuntimeError as e:
                if 'out of memory' in str(e).lower() and batch > 4:
                    print('  OOM at batch %d -> retry at 4' % batch)
                    torch.cuda.empty_cache()
                    _do(4)
                else:
                    raise
    except KeyboardInterrupt:
        print('  %s finetune_768 interrupted - re-run the cell to resume from last.pt' % pretty)
        return None
    print('  %s finetune_768 finished in %.0f min' % (pretty, (time.time() - t0) / 60))
    val = _validate_and_record(pretty, 'finetune_768', subdir, 'finetune_768', 768, epochs, 0.0002, 12)
    _compare_to_baseline(pretty, subdir, float(val.box.map50))
    return run_dir / 'weights' / 'best.pt'


def _compare_to_baseline(pretty, subdir, ft_map50):
    """Append a one-line note on whether 768 improved over the 640 baseline (validation mAP@0.5)."""
    import pandas as pd
    path = RUNS_DIR / 'experiment_tracker.csv'
    if not path.exists():
        return
    df = pd.read_csv(path)
    base = df[(df['model'] == pretty) & (df['stage'] == 'baseline_640')]
    if base.empty:
        return
    b = float(base.iloc[-1]['val_mAP50'])
    verdict = 'improved' if ft_map50 > b else 'did not improve'
    append_training_notes('  -> %s 768 fine-tune %s over 640 (val mAP@0.5 %.4f vs %.4f)'
                          % (pretty, verdict, ft_map50, b))

## Part 2.2 YOLOv11s baseline_640

Trains `baseline_640` (640px, 110 epochs, patience=25, AdamW, cosine LR). Checkpoints every 5 epochs; re-run to resume from `last.pt`. Validation metrics go to `runs/experiment_tracker.csv`; test is scored only in Part 3. Runs when `RUN_YOLO11S=True`.

**Note:** the 768 fine-tune stage was tested on v2 and v3 but did not improve validation mAP (v3: finetune_768 0.441 vs baseline_640 0.461). Available as optional ablation via `RUN_FINETUNE_768=True` but not part of the default workflow.

In [ ]:
if RUN_YOLO11S:
    _best = train_baseline_640('yolo11s.pt', 'yolo11s', 'YOLOv11s')
    # 768 fine-tune was tested on v2/v3 and regressed vs baseline_640.
    # Set RUN_FINETUNE_768=True only for ablation experiments.
    if _best is not None and RUN_FINETUNE_768:
        finetune_768('yolo11s', 'YOLOv11s')
else:
    print('RUN_YOLO11S=False - skipping YOLOv11s')

## Part 2.3 YOLOv8s baseline_640

Same `baseline_640` flow as Part 2.2 but for YOLOv8s, in its own independent folders (`runs/yolov8s/baseline_640`). **Independent of YOLOv11s** — runs whenever `RUN_YOLO8S=True`.

In [ ]:
# Independent of YOLOv11s: own folders (runs/yolov8s/...), own resume, no YOLOv11s dependency.
if RUN_YOLO8S:
    _best8 = train_baseline_640('yolov8s.pt', 'yolov8s', 'YOLOv8s')
    # 768 fine-tune disabled by default - set RUN_FINETUNE_768=True for ablation only.
    if _best8 is not None and RUN_FINETUNE_768:
        finetune_768('yolov8s', 'YOLOv8s')
else:
    print('RUN_YOLO8S=False - skipping YOLOv8s')

## Part 2.4 SSDLite320-MobileNetV3 (optional - set RUN_SSDLITE=True) (anchor-based, ImageNet backbone)

The old SSDLite scored ~0 because (a) torchvision reserves **class 0 for background** — labels
must be shifted `+1` and `num_classes = 6` — and (b) the default COCO anchors (aspect ratios
{1/3..3}) cannot match the long, narrow shapes in this data (boxes span {0.19..6.7}).

Fixes applied here:
- **labels +1, num_classes = 6** (class 0 = background),
- **custom anchors** `ANCHOR_AR = (2, 3, 5)` → boxes {1/5..5}, head rebuilt to match,
- **two-phase SGD** (freeze backbone epochs 1–5, then unfreeze), momentum 0.9, lr 5e-4.

SSDLite is still expected to trail YOLO — it is the lightweight edge/mobile comparison point,
not the accuracy winner.

In [ ]:
# 2.3a SSDLite builder + YOLO-format dataset (labels shifted +1 for background=0)
import torchvision.transforms.functional as TF
from functools import partial
from torch.utils.data import DataLoader, Dataset
from torchvision.models.detection import ssdlite320_mobilenet_v3_large
from torchvision.models.detection import _utils as det_utils
from torchvision.models.detection.anchor_utils import DefaultBoxGenerator
from torchvision.models.detection.ssdlite import SSDLiteHead
from torchvision.models import MobileNet_V3_Large_Weights
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from tqdm.auto import tqdm

def build_ssd(num_classes=NUM_CLASSES, pretrained=True, anchor_aspect_ratios=ANCHOR_AR, img_size=SSD_IMG):
    """SSDLite320-MobileNetV3. anchor_aspect_ratios=(2,3,5) widens anchors to {1/5..5}
    (default {1/3..3}) for long/narrow defects; the head is rebuilt to match. img_size>320
    retunes the internal transform to that resolution."""
    model = ssdlite320_mobilenet_v3_large(
        weights=None, weights_backbone=MobileNet_V3_Large_Weights.IMAGENET1K_V2,
        num_classes=num_classes)
    if anchor_aspect_ratios is not None:
        out_channels = det_utils.retrieve_out_channels(model.backbone, (img_size, img_size))
        ar = list(anchor_aspect_ratios)
        ag = DefaultBoxGenerator([ar for _ in range(len(out_channels))], min_ratio=0.2, max_ratio=0.95)
        norm_layer = partial(torch.nn.BatchNorm2d, eps=0.001, momentum=0.03)
        model.anchor_generator = ag
        model.head = SSDLiteHead(out_channels, ag.num_anchors_per_location(), num_classes, norm_layer)
    if img_size != 320:
        model.transform.min_size = (img_size,)
        model.transform.max_size = img_size
        model.transform.fixed_size = (img_size, img_size)
    if not pretrained:
        import torch.nn as nn
        for m in model.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, 0.0, 0.03)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(m.weight, 1.0)
                nn.init.constant_(m.bias, 0.0)
    return model

class DefectDataset(Dataset):
    def __init__(self, split, img_size=SSD_IMG):
        self.img_size = img_size
        self.img_dir = DATA_DIR / 'images' / split
        self.lbl_dir = DATA_DIR / 'labels' / split
        self.images = sorted(p for p in self.img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        p = self.images[idx]; S = self.img_size
        try:
            img = Image.open(p).convert('RGB').resize((S, S))
        except Exception:
            img = Image.new('RGB', (S, S))
        tensor = TF.to_tensor(img)
        boxes, labels = [], []
        lbl = self.lbl_dir / (p.stem + '.txt')
        if lbl.exists():
            for line in lbl.read_text().splitlines():
                parts = line.split()
                if len(parts) != 5:
                    continue
                cls, cx, cy, bw, bh = int(parts[0]), *map(float, parts[1:])
                x1 = max(0.0, (cx - bw / 2) * S); y1 = max(0.0, (cy - bh / 2) * S)
                x2 = min(float(S), (cx + bw / 2) * S); y2 = min(float(S), (cy + bh / 2) * S)
                if x2 > x1 and y2 > y1:
                    boxes.append([x1, y1, x2, y2]); labels.append(cls + 1)   # +1: 0 = background
        if boxes:
            bt = torch.tensor(boxes, dtype=torch.float32)
            lt = torch.tensor(labels, dtype=torch.int64)
            area = (bt[:, 3] - bt[:, 1]) * (bt[:, 2] - bt[:, 0])
        else:
            bt = torch.zeros((0, 4), dtype=torch.float32)
            lt = torch.zeros(0, dtype=torch.int64)
            area = torch.zeros(0, dtype=torch.float32)
        target = {'boxes': bt, 'labels': lt, 'image_id': torch.tensor([idx]),
                  'area': area, 'iscrowd': torch.zeros(len(lt), dtype=torch.int64)}
        return tensor, target

def collate_fn(batch):
    return tuple(zip(*batch))

ssd_train = DataLoader(DefectDataset('train'), batch_size=SSD_BATCH, shuffle=True,
                       collate_fn=collate_fn, num_workers=WORKERS, drop_last=True)
ssd_val = DataLoader(DefectDataset('val'), batch_size=SSD_BATCH, shuffle=False,
                     collate_fn=collate_fn, num_workers=WORKERS)
print('SSD loaders ready (img=%d, batch=%d, anchors_ar=%s).' % (SSD_IMG, SSD_BATCH, ANCHOR_AR))

In [ ]:
if RUN_SSDLITE:
    # 2.3b SSDLite training — custom anchors + two-phase SGD (same EPOCHS as YOLO)
    @torch.no_grad()
    def ssd_evaluate(model, loader):
        model.eval()
        metric = MeanAveragePrecision(iou_type='bbox')
        for images, targets in tqdm(loader, desc='eval', leave=False):
            imgs = [i.to(DEVICE) for i in images]
            outs = model(imgs)
            metric.update(
                [{'boxes': o['boxes'].cpu(), 'scores': o['scores'].cpu(), 'labels': o['labels'].cpu()} for o in outs],
                [{'boxes': t['boxes'], 'labels': t['labels']} for t in targets])
        torch.cuda.empty_cache()
        return metric.compute()

    ssd_ver = 1
    (RUNS_DIR / 'ssdlite').mkdir(parents=True, exist_ok=True)
    while (RUNS_DIR / 'ssdlite' / ('best_v%d.pth' % ssd_ver)).exists():
        ssd_ver += 1
    best_ckpt = RUNS_DIR / 'ssdlite' / ('best_v%d.pth' % ssd_ver)

    ssd = build_ssd(num_classes=NUM_CLASSES, pretrained=True).to(DEVICE)
    ssd_params = sum(p.numel() for p in ssd.parameters()) / 1e6
    print('SSDLite %.1fM params, anchors_ar=%s, img=%d -> %s' % (ssd_params, ANCHOR_AR, SSD_IMG, best_ckpt.name))

    # Phase 1: freeze backbone, train detection head only
    for p in ssd.backbone.parameters():
        p.requires_grad = False
    opt = torch.optim.SGD([p for p in ssd.parameters() if p.requires_grad],
                          lr=SSD_LR, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=5)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

    best_map = 0.0
    t0 = time.time()
    for epoch in range(1, EPOCHS + 1):
        if epoch == 6:   # Phase 2: unfreeze backbone, lower LR for fine-tuning
            for p in ssd.backbone.parameters():
                p.requires_grad = True
            opt = torch.optim.SGD(ssd.parameters(), lr=SSD_LR / 5, momentum=0.9, weight_decay=5e-4)
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(EPOCHS - 5, 1))
        ssd.train()
        running = 0.0
        for images, targets in tqdm(ssd_train, desc='ep%d' % epoch, leave=False):
            imgs = [i.to(DEVICE) for i in images]
            tgts = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
            opt.zero_grad()
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                loss = sum(ssd(imgs, tgts).values())
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            running += loss.item()
        sched.step()
        metrics = ssd_evaluate(ssd, ssd_val)
        m50 = float(metrics['map_50'])
        print('  epoch %3d/%d  loss=%.4f  mAP@0.5=%.4f' % (epoch, EPOCHS, running / len(ssd_train), m50))
        if m50 > best_map:
            best_map = m50
            torch.save({'epoch': epoch, 'model_state_dict': ssd.state_dict(),
                        'anchor_ar': ANCHOR_AR, 'img_size': SSD_IMG, 'metrics': metrics}, best_ckpt)

    ssd_min = (time.time() - t0) / 60
    final = ssd_evaluate(ssd, ssd_val)
    log_run('SSDLite', EPOCHS, SSD_BATCH, SSD_IMG, 'SGD', float(final['map_50']), float(final['map']),
            0.0, 0.0, ssd_min, ssd_params, str(best_ckpt),
            notes='custom anchors {1/5..5}, two-phase SGD mom0.9, lr=%.0e' % SSD_LR)
    print('SSDLite best mAP@0.5=%.4f (%.0f min)' % (best_map, ssd_min))
else:
    print('RUN_SSDLITE=False - skipping SSDLite training')

## Part 3 Final test evaluation

Scores the **test set once** for each trained model and writes `runs/model_comparison.csv`. Uses `baseline_640` as the selected stage (confirmed best on v3 validation). Falls back to best-by-validation-mAP if the tracker has a higher-scoring alternative. Run after Parts 2.2 / 2.3. Never mix test rows into `experiment_tracker.csv`.

In [ ]:
# Part 3 Final test evaluation.
# Default: scores baseline_640 for each model (the confirmed best stage on v3).
# Selection logic: picks baseline_640 by default; falls back to best-by-val only if a
# different stage actually outperforms it. The test set is scored ONCE per model.
# Writes runs/model_comparison.csv (test only - never mixed with the validation tracker).
import os as _os
import time as _time
import numpy as _np
import pandas as _pd
import matplotlib.pyplot as _plt
from ultralytics import YOLO as _YOLO

_SUBDIR = {'YOLOv11s': 'yolo11s', 'YOLOv8s': 'yolov8s'}


def evaluate_on_test():
    tracker = RUNS_DIR / 'experiment_tracker.csv'
    out = []

    # Determine which models to evaluate: prefer experiment_tracker entries,
    # fall back to scanning for baseline_640 weights directly.
    candidates = {}
    if tracker.exists():
        val_df = _pd.read_csv(tracker)
        # Filter to current dataset version if column exists
        if 'dataset_version' in val_df.columns:
            val_df = val_df[val_df['dataset_version'].astype(str) == str(DATASET_VERSION)]
        for model in sorted(val_df['model'].unique()):
            sub = val_df[val_df['model'] == model]
            best = sub.loc[sub['val_mAP50'].idxmax()]
            candidates[model] = (best['stage'], int(best['imgsz']), float(best['val_mAP50']))
    # Also scan for baseline_640 weights not yet in tracker
    for model, subdir in _SUBDIR.items():
        if model not in candidates:
            w = RUNS_DIR / subdir / 'baseline_640' / 'weights' / 'best.pt'
            if w.exists():
                candidates[model] = ('baseline_640', 640, 0.0)

    if not candidates:
        print('no trained models found - run Part 2.2 / 2.3 first')
        return

    for model, (stage, imgsz, val_map) in sorted(candidates.items()):
        weights = RUNS_DIR / _SUBDIR.get(model, model.lower()) / stage / 'weights' / 'best.pt'
        if not weights.exists():
            print('  %s: weights missing (%s) - skip' % (model, weights))
            continue
        note = '' if val_map == 0.0 else ' (val mAP@0.5=%.4f)' % val_map
        print('  %s: stage=%s imgsz=%d%s -> scoring TEST' % (model, stage, imgsz, note))
        m = _YOLO(str(weights))
        v = m.val(data=DATA_YAML, split='test', imgsz=imgsz)
        p, r = float(v.box.mp), float(v.box.mr)
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        imgs = [q for q in (DATA_DIR / 'images' / 'test').iterdir()
                if q.suffix.lower() in IMAGE_EXTS][:50]
        ts = []
        for q in imgs:
            t0 = _time.time(); m.predict(str(q), imgsz=imgsz, verbose=False)
            ts.append((_time.time() - t0) * 1000)
        fps = round(1000 / _np.mean(ts), 1) if ts else 0.0
        out.append({'dataset_version': DATASET_VERSION, 'model': model,
                    'chosen_stage': stage, 'imgsz': imgsz,
                    'test_mAP50': round(float(v.box.map50), 4),
                    'test_mAP50_95': round(float(v.box.map), 4),
                    'precision': round(p, 4), 'recall': round(r, 4), 'f1': round(f1, 4),
                    'fps': fps, 'size_MB': round(_os.path.getsize(weights) / 1e6, 1),
                    'weights': weights.as_posix()})
        try:
            per = dict(zip(CLASSES, [round(float(x), 4) for x in v.box.maps]))
            fig, ax = _plt.subplots(figsize=(8, 4))
            ax.bar(list(per.keys()), list(per.values()), color=CLASS_COLORS, edgecolor='black')
            ax.set_title('%s (%s) - Per-Class AP@0.5:0.95 (test)' % (model, stage))
            ax.set_ylim(0, 1)
            for i, (k, val) in enumerate(per.items()):
                ax.text(i, val + 0.01, '%.3f' % val, ha='center', fontsize=9)
            _plt.tight_layout()
            _plt.savefig(RUNS_DIR / ('per_class_ap_test_%s.png' % model.lower()), dpi=150)
            _plt.show()
        except Exception as e:
            print('  (per-class chart skipped: %s)' % e)
    if not out:
        print('no models evaluated')
        return
    comp = _pd.DataFrame(out)
    comp.to_csv(RUNS_DIR / 'model_comparison.csv', index=False)
    print('\nFinal test results [dataset=%s] -> runs/model_comparison.csv' % DATASET_VERSION)
    print(comp[['model', 'chosen_stage', 'test_mAP50', 'test_mAP50_95',
                'precision', 'recall', 'f1', 'fps']].to_string(index=False))
    print('\nNote: model_comparison.csv = TEST only; experiment_tracker.csv = VALIDATION only.')


evaluate_on_test()

## Optional (set RUN_RTDETR=True) - RT-DETR (Real-Time Detection Transformer)

A 4th comparison model: a **transformer-based** real-time detector (Ultralytics `RTDETR`, ~32M params). Unlike the YOLO family (convolutional, anchor-free) and SSDLite (anchor-based), RT-DETR uses **transformer object queries** for end-to-end detection (no NMS). It is trained on the **same** data.yaml, train/val/test split, image size (640) and epoch budget as the others for a fair comparison. Labels stay `0..4` - the `+1` background shift is **only** for torchvision SSDLite, never for RT-DETR. This cell is self-contained: train -> evaluate on the unseen test set -> per-class AP -> confidence sweep -> prediction samples -> merge into `runs/model_comparison.csv`.

In [ ]:
if RUN_RTDETR:
    # ============ RT-DETR (transformer detector) - train + evaluate (self-contained) ============
    # Runs AFTER Part 3 so runs/model_comparison.csv already exists for the other 3 models.
    # Uses the SAME data.yaml, train/val/test split, image size and epoch budget for a FAIR
    # comparison. Labels stay 0..4 (NO +1 shift - that is only for torchvision SSDLite).
    # Trains -> evaluates on the UNSEEN test set -> per-class AP -> confidence sweep ->
    # prediction samples -> MERGES its row into runs/model_comparison.csv.
    from ultralytics import RTDETR
    import os, time
    import numpy as np, pandas as pd
    import matplotlib.pyplot as plt, matplotlib.patches as patches
    from PIL import Image

    # ---- RT-DETR config (self-contained; reuses EPOCHS/IMG_SIZE/BIG_GPU from Part 0) ----
    RTDETR_WEIGHTS = 'rtdetr-l.pt'              # smallest Ultralytics RT-DETR (~32M params)
    RTDETR_EPOCHS  = EPOCHS                      # SAME budget as YOLO/SSD (fair comparison)
    RTDETR_IMG     = IMG_SIZE                    # 640, same as YOLO
    RTDETR_BATCH   = 4 if BIG_GPU else 2         # heavier than YOLO; raise to 8 only if VRAM allows,
                                                 # drop to 2 on CUDA OOM (16->8->4->2)
    RTDETR_LR      = 1e-4                        # DETR-style models prefer a lower LR than YOLO
    print('RT-DETR config: %s epochs=%d imgsz=%d batch=%d lr=%.0e'
          % (RTDETR_WEIGHTS, RTDETR_EPOCHS, RTDETR_IMG, RTDETR_BATCH, RTDETR_LR))

    # ---- 2.4  TRAIN (auto-versioned, never overwrites existing runs) ----
    _sub = 'rtdetr'
    _ver = next_version(_sub)
    print('RT-DETR -> runs/%s/v%d  (epochs=%d, imgsz=%d, batch=%d)'
          % (_sub, _ver, RTDETR_EPOCHS, RTDETR_IMG, RTDETR_BATCH))
    rtdetr = RTDETR(RTDETR_WEIGHTS)                      # downloads rtdetr-l.pt (~63 MB) if absent
    _t0 = time.time()
    rtdetr.train(data=DATA_YAML, epochs=RTDETR_EPOCHS, imgsz=RTDETR_IMG, batch=RTDETR_BATCH,
                 optimizer='AdamW', lr0=RTDETR_LR, cos_lr=True,
                 device=0 if torch.cuda.is_available() else 'cpu',
                 project=str(RUNS_DIR / _sub), name='v%d' % _ver, exist_ok=False,
                 amp=USE_AMP, workers=WORKERS, plots=True, verbose=True)
    _mins = (time.time() - _t0) / 60
    _wts = RUNS_DIR / _sub / ('v%d' % _ver) / 'weights' / 'best.pt'
    print('RT-DETR trained in %.0f min -> %s' % (_mins, _wts))

    # ---- 3.x  EVALUATE on the UNSEEN test set ----
    rtdetr_best = RTDETR(str(_wts))
    val = rtdetr_best.val(data=DATA_YAML, split='test', imgsz=RTDETR_IMG)
    p, r = float(val.box.mp), float(val.box.mr)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    params = sum(pp.numel() for pp in rtdetr_best.model.parameters()) / 1e6
    _imgs = [q for q in (DATA_DIR / 'images' / 'test').iterdir()
             if q.suffix.lower() in IMAGE_EXTS][:50]
    _ts = []
    for q in _imgs:
        _a = time.time(); rtdetr_best.predict(str(q), verbose=False); _ts.append((time.time() - _a) * 1000)
    _fps = round(1000 / np.mean(_ts), 1) if _ts else 0.0
    row = {'Epochs': RTDETR_EPOCHS, 'Image Size': RTDETR_IMG, 'Optimizer': 'AdamW',
           'Params (M)': round(params, 2),
           'mAP@0.5': round(float(val.box.map50), 4), 'mAP@0.5:0.95': round(float(val.box.map), 4),
           'Precision': round(p, 4), 'Recall': round(r, 4), 'F1': round(f1, 4),
           'FPS': _fps, 'Size (MB)': round(os.path.getsize(_wts) / 1e6, 1),
           'Notes': 'transformer (DETR), AdamW, cosine LR'}
    print('RT-DETR test: mAP@0.5=%.4f mAP@0.5:0.95=%.4f P=%.3f R=%.3f F1=%.3f FPS=%.1f'
          % (row['mAP@0.5'], row['mAP@0.5:0.95'], p, r, f1, _fps))
    log_run('RT-DETR', RTDETR_EPOCHS, RTDETR_BATCH, RTDETR_IMG, 'AdamW',
            row['mAP@0.5'], row['mAP@0.5:0.95'], p, r, _mins, params, str(_wts),
            notes='RT-DETR-l transformer, lr=%.0e' % RTDETR_LR)

    # ---- MERGE into runs/model_comparison.csv (other 3 models already written by Part 3) ----
    _cmp = RUNS_DIR / 'model_comparison.csv'
    comp = pd.read_csv(_cmp, index_col=0) if _cmp.exists() else pd.DataFrame()
    comp.loc['RT-DETR'] = pd.Series(row)
    comp.to_csv(_cmp)
    print('\nUpdated', _cmp)
    print(comp.to_string())

    # ---- PER-CLASS AP ----
    per = dict(zip(CLASSES, [round(float(x), 4) for x in val.box.maps]))
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(list(per.keys()), list(per.values()), color=CLASS_COLORS, edgecolor='black')
    ax.set_title('RT-DETR - Per-Class AP@0.5:0.95'); ax.set_ylim(0, 1)
    for i, (k, v) in enumerate(per.items()):
        ax.text(i, v + 0.01, '%.3f' % v, ha='center', fontsize=9)
    plt.tight_layout(); plt.savefig(RUNS_DIR / 'per_class_ap_rtdetr.png', dpi=150); plt.show()

    # ---- CONFIDENCE SWEEP (recall is the project's weak spot -> find the operating point) ----
    rows = []
    for t in [0.05, 0.10, 0.15, 0.20, 0.25]:
        v = rtdetr_best.val(data=DATA_YAML, split='test', imgsz=RTDETR_IMG, conf=t, verbose=False)
        pp, rr = float(v.box.mp), float(v.box.mr)
        rows.append({'conf': t, 'precision': round(pp, 4), 'recall': round(rr, 4),
                     'f1': round(2 * pp * rr / (pp + rr) if (pp + rr) > 0 else 0.0, 4),
                     'mAP@0.5': round(float(v.box.map50), 4),
                     'mAP@0.5:0.95': round(float(v.box.map), 4)})
    sweep = pd.DataFrame(rows); sweep.to_csv(RUNS_DIR / 'rtdetr_conf_threshold_sweep.csv', index=False)
    print('\nRT-DETR confidence sweep (test):'); print(sweep.to_string(index=False))
    fig, ax = plt.subplots(figsize=(7, 5))
    for col in ('precision', 'recall', 'f1'):
        ax.plot(sweep['conf'], sweep[col], 'o-', label=col.capitalize())
    ax.set_xlabel('Confidence threshold'); ax.set_ylabel('Score')
    ax.set_title('RT-DETR confidence sweep (test)'); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(RUNS_DIR / 'rtdetr_conf_threshold_sweep.png', dpi=120); plt.show()

    # ---- QUALITATIVE PREDICTIONS (same first 6 unseen test images as the YOLO samples) ----
    _test6 = sorted(q for q in (DATA_DIR / 'images' / 'test').iterdir()
                    if q.suffix.lower() in IMAGE_EXTS)[:6]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8)); axes = axes.flatten()
    for i, q in enumerate(_test6):
        res = rtdetr_best.predict(str(q), conf=0.25, verbose=False)[0]
        im = Image.open(q).convert('RGB'); axes[i].imshow(im); w, h = im.size
        for box in res.boxes:
            c = int(box.cls.item()); cf = float(box.conf.item())
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            s = severity_label((x2 - x1) / w, (y2 - y1) / h)
            axes[i].add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, lw=2,
                              edgecolor=CLASS_COLORS[c], facecolor='none'))
            axes[i].text(x1, y1 - 5, '%s %.2f [%s]' % (CLASSES[c], cf, s),
                         color=CLASS_COLORS[c], fontsize=7, fontweight='bold')
        axes[i].axis('off'); axes[i].set_title(q.name, fontsize=8)
    plt.suptitle('RT-DETR Predictions with Severity (unseen test)')
    plt.tight_layout(); plt.savefig(RUNS_DIR / 'prediction_samples_rtdetr.png', dpi=120); plt.show()
    print('\nRT-DETR done. Compare in runs/model_comparison.csv (now 4 models).')
else:
    print('RUN_RTDETR=False - skipping RT-DETR')

## Part 4 — Summary, Trade-offs & Report Notes

**Why the new numbers are honest.** The old v1 YOLO scored ~0.76 mAP@0.5 because some classes
used whole-image *proxy* boxes (any overlapping prediction counted as correct — it measured
*presence*, not *localisation*). With real localised boxes the honest baseline is ~0.44; every
improvement below is measured against that.

**What drives the improvement.**
1. `s` models (YOLOv11s/YOLOv8s) have far more capacity than nano → resolve thin, irregular
   defect texture.
2. 110 epochs + AdamW + cosine LR → fuller, smoother convergence.
3. Mosaic / affine / HSV augmentation → robustness to lighting, scale and surface variation
   (train split only — val/test are never augmented).
4. Clean labels (degenerate boxes dropped, leakage removed) → the model learns correct targets.
5. SSDLite fixed: labels shifted **+1** (`num_classes = 6`, background = 0) and **custom anchors**
   {1/5..5} that match long/narrow defects — lifting it off the near-zero floor.

**Trade-offs (for the report / interview).**
- *Capacity vs deployment:* `s` models are more accurate but ~3–4× larger and slower than nano —
  fine on a workstation, marginal on edge devices.
- *Epochs vs overfitting:* 110 epochs help the larger models converge but risk memorising a
  ~6k-image set; best-checkpoint selection + watching val mAP guards this.
- *Optimiser:* AdamW (YOLO) = faster convergence; SGD+momentum (SSD) = the stable standard.
- *Augmentation:* stronger aug improves generalisation but can fabricate unrealistic samples —
  kept moderate, mosaic closed for the last 20 epochs.
- *Anchors:* custom anchors lift SSD recall on elongated defects but are dataset-specific.

**Expected ranking:** YOLOv11s ≈ YOLOv8s ≫ SSDLite. SSDLite is included as the lightweight
edge/mobile comparison, not the accuracy winner — even fixed, its anchor-based design and small
backbone trail YOLO on texture-like defects. Do not fake numbers: if SSDLite stays low, that is
the expected architectural story.

In [ ]:
# Zip results to /kaggle/working so they appear in the Output tab.
# Click 'Save Version' (top-right) BEFORE the session ends to commit the output.
import zipfile, os

zip_path = '/kaggle/working/runs_results.zip'
print('Zipping runs/ to', zip_path, '...')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
    for root, dirs, files in os.walk(str(RUNS_DIR)):
        for file in files:
            fp = os.path.join(root, file)
            arcname = os.path.relpath(fp, str(RUNS_DIR.parent))
            try:
                zf.write(fp, arcname)
            except Exception as e:
                print('  Skipped:', arcname, '(%s)' % e)
size_mb = os.path.getsize(zip_path) / 1e6
print('Done: %s  (%.1f MB)' % (zip_path, size_mb))
print()
print('=' * 60)
print('>>> CLICK Save Version (top-right) NOW to commit output <<<')
print('=' * 60)